## KServe (Experimentell )

![](https://kserve.github.io/website/assets/images/architecture_overview-68f636ea2d016e39031096ba409d4190.png)

Quelle: [Kserve](https://kserve.github.io/website/docs/concepts/architecture/control-plane-llmisvc)

- - -

KServe ist eine Kubernetes-native Plattform, mit der Machine-Learning- und LLM-Modelle standardisiert, skalierbar und API-basiert als Inference Services bereitgestellt werden.

### Installation

Zuerst muss eine etwaige Cert-Manager Installation entfernt werden

In [ ]:
%%bash
kubectl delete -f https://github.com/cert-manager/cert-manager/releases/download/v1.20.2/cert-manager.yaml

Dann kann mit der eigentlichen Installation von `kserve` begonnen werden.

In [ ]:
%%bash
git clone https://github.com/kserve/kserve.git
cd kserve    
./hack/kserve-install.sh --kserve-version v0.18.0 --type kserve,localmodel --standard

Nach der Installation muss eine neue Custom Resource `InferenceService` vorhanden sein

In [ ]:
%%bash
kubectl explain InferenceService

---

### LLM Model Deployen



In [ ]:
%%bash
kubectl create namespace kserve-test --dry-run=client -o yaml | kubectl apply -f -
kubectl apply -f - <<EOF
apiVersion: serving.kserve.io/v1beta1
kind: InferenceService
metadata:
  name: qwen-llm
  namespace: kserve-test
  annotations:
    serving.kserve.io/deploymentMode: RawDeployment
spec:
  predictor:
    minReplicas: 1
    affinity:
      nodeAffinity:
        requiredDuringSchedulingIgnoredDuringExecution:
          nodeSelectorTerms:
          - matchExpressions:
            - key: nvidia.com/gpu.product
              operator: In
              values:
              - NVIDIA-GeForce-RTX-3050-Ti-Laptop-GPU-SHARED
    model:
      modelFormat:
        name: huggingface
      args:
        - --model_name=qwen
        - --backend=huggingface
        - --dtype=float16
        - --max_model_len=1024
      storageUri: hf://Qwen/Qwen2.5-0.5B-Instruct
      resources:
        requests:
          cpu: "1"
          memory: 4Gi
          nvidia.com/gpu: "1"
        limits:
          cpu: "2"
          memory: 6Gi
          nvidia.com/gpu: "1"
EOF


Wir checken den Status

In [ ]:
%%bash
kubectl get inferenceservices qwen-llm -n kserve-test
kubectl get all -n kserve-test

Da wir den erstellen Service nicht ändern können, erstellen wir einen neuen

In [ ]:
%%bash
kubectl apply -n kserve-test -f - <<EOF
apiVersion: v1
kind: Service
metadata:
  name: qwen-llm-lb
spec:
  type: LoadBalancer
  selector:
    serving.kserve.io/inferenceservice: qwen-llm
  ports:
    - name: http
      port: 80
      targetPort: 8080
      protocol: TCP
EOF

---

### Testen

Über dessen NodePort können wir nun auf das LLM zugreifen

In [ ]:
%%bash
source ~/data/env.py
cat <<EOF | tee ~/data/env-kserve-nvidia.py
OPENAI_API_KEY="kserve-nvidia"
HF_TOKEN=""
AI_KUBECONFIG="$AI_KUBECONFIG"
AI_MODEL="qwen"
AI_NAME=""
AI_IP="${AI_IP}"
AI_BASE_URL='http://localhost:$(kubectl get svc qwen-llm-lb -n kserve-test -o jsonpath='{.spec.ports[?(@.name=="http")].nodePort}')/openai/v1'
EOF

In [ ]:
%run ~/data/env-kserve-nvidia.py
from openai import OpenAI

client = OpenAI(
    base_url=AI_BASE_URL,
    api_key="dummy",
)

response = client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "user", "content": "Antworte in einem Satz: Wer war John F. Kennedy?"}
    ],
    max_tokens=80,
    temperature=0.2,
)

print(response.choices[0].message.content)

---

### Aufräumen

In [ ]:
%%bash
kubectl delete inferenceservices qwen-llm -n kserve-test